# M1 lab — search Hub models with a wildcard and estimate memory

This notebook builds a small candidate list from the Hugging Face Hub, filters repository IDs with a shell-style wildcard, and runs Accelerate’s official `estimate-memory` command for each match.

The Hub API accepts a text search, not a wildcard expression. The notebook therefore uses two steps: first ask the Hub for a bounded set of likely candidates, then apply Python’s `fnmatch` wildcard matching to their full repository IDs. The limit and sort order matter: a wildcard only sees candidates returned by the Hub query.

The estimator reads model configuration and constructs an empty model on the `meta` device; it does not download model weights. Its table estimates loading sizes at different dtypes and an approximate training column. It does not include inference KV cache or guarantee a particular quantized runtime file will fit.

## 1. Install and import the tools

Use Python 3.10 or later. In a local environment, install `m1/requirements-memory.txt`. The first cell can also install the two libraries when run in Colab.

In [ ]:
import importlib.util
import subprocess
import sys

missing = [name for name in ('huggingface_hub', 'accelerate')
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'huggingface_hub>=0.24', 'accelerate>=1.4'])

from fnmatch import fnmatchcase
import shutil
from huggingface_hub import HfApi
print('Python:', sys.version.split()[0])
print('Accelerate CLI:', shutil.which('accelerate') or 'will use its Python module entry point')

## 2. Search the Hub, then apply a wildcard

Change `HUB_SEARCH` to a short substring that narrows the server-side search. Change `MODEL_PATTERN` to a wildcard over complete repository IDs. `*` matches any sequence of characters; `?` matches one character. Here, the Hub search finds Qwen3 repositories and `Qwen/Qwen3-*` keeps the Qwen organization’s Qwen3 model IDs.

In [ ]:
HUB_SEARCH = 'Qwen3'
MODEL_PATTERN = 'Qwen/Qwen3-*'
CANDIDATE_LIMIT = 100

api = HfApi()
candidates = list(api.list_models(
    search=HUB_SEARCH,
    filter='transformers',
    sort='downloads',
    limit=CANDIDATE_LIMIT,
))
matched = [model for model in candidates if fnmatchcase(model.id, MODEL_PATTERN)]
print(f'Hub returned {len(candidates)} candidates; {len(matched)} match {MODEL_PATTERN!r}.')
print('The Hub search and result limit define the pool searched by the wildcard.')
assert matched, 'No repository IDs matched; broaden HUB_SEARCH or adjust MODEL_PATTERN.'

### Inspect the shortlist

Downloads provide a rough popularity signal, not a quality score. Check each model card, licence, architecture and supported runtime before adopting a candidate.

In [ ]:
import pandas as pd

shortlist = pd.DataFrame([
    {'model_id': model.id, 'downloads': model.downloads or 0,
     'likes': model.likes or 0, 'pipeline': model.pipeline_tag or '—'}
    for model in matched
]).sort_values('downloads', ascending=False, ignore_index=True)
display(shortlist.head(20))

## 3. Estimate memory for selected matches

The official command accepts a Hub model ID. Choose a few matches instead of estimating every result. Accelerate reports each selected dtype’s largest layer and total loading size; its `Training using Adam` column is a separate approximate training estimate.

In [ ]:
DTYPES = ['float16', 'int8', 'int4']
MAX_ESTIMATES = 3
selected_ids = shortlist['model_id'].head(MAX_ESTIMATES).tolist()
print('Selected for estimates:', selected_ids)

In [ ]:
def estimate_memory(model_id, dtypes=DTYPES):
    executable = shutil.which('accelerate')
    if executable:
        command = [executable, 'estimate-memory', model_id,
                   '--library_name', 'transformers', '--dtypes', *dtypes]
    else:
        command = [sys.executable, '-m', 'accelerate.commands.estimate', model_id,
                   '--library_name', 'transformers', '--dtypes', *dtypes]
    result = subprocess.run(command, text=True, capture_output=True, timeout=240)
    if result.returncode:
        return f'ESTIMATE FAILED for {model_id}:\n{result.stderr.strip()}'
    return result.stdout.strip()

for model_id in selected_ids:
    print('\n' + '=' * 88)
    print(estimate_memory(model_id))

## 4. Interpret the estimate

Use `Total Size` as an initial estimate for the selected dtype. Leave room for runtime overhead, activations and the KV cache. The KV cache depends on architecture, context length, batch/concurrency and KV precision; estimate it separately (see M3). Actual GGUF, GPTQ, AWQ or other quantized files can differ from Accelerate’s idealized dtype sizes. Measure the exact artifact in the runtime you plan to deploy.

If a model is gated or private, authenticate with the Hugging Face CLI before running the notebook. Do not paste access tokens into notebook cells or commit them. If a repository uses custom model code, inspect and trust that code before adding `--trust_remote_code` to the estimator command.

### Try another search

1. Change `HUB_SEARCH` and `MODEL_PATTERN` to a different family.
2. Increase `CANDIDATE_LIMIT` if the search pool is too small; the wildcard is applied only after this limit.
3. Compare `float16`, `int8` and `int4`, then add an independent KV-cache estimate and a safety margin.
4. Record the exact model ID, estimator output, date, runtime and quantized file in the M1 selection worksheet.